# Cum am construit `app/app.py` — tutorial Gradio (pas cu pas)

Ideea Gradio, într-o propoziție: **o funcție Python devine o interfață web.**
Tu scrii funcția, Gradio face caseta, butonul și layout-ul.

Construim app-ul incremental, exact în ordinea în care e scris `app.py`:
funcția simplă -> mai multe input-uri -> tab Chat -> starea partajată ->
regula subiect/știre -> tab Agent -> punem tab-urile împreună -> recapitulare.

Inspirat din [Gradio Quickstart](https://www.gradio.app/guides/quickstart).
Regula tutorialului: **cât mai simplu, doar esențialul.**

> `app/app.py` este doar un strat subțire de Gradio peste `core/` (agent, graph),
> construit în cursurile C2–C7. Aici nu rescriem `core/` — îl chemăm.
> Ca să ruleze fără chei API, folosim un backend fals.

In [1]:
# o singură dată:  %pip install -q gradio
import gradio as gr
print("Gradio", gr.__version__)

/Users/lostunflaviu/Documents/Ingineria AI/echochamber-project-team3/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio 6.14.0


## 1. Cel mai simplu Gradio

`gr.Interface` are nevoie de 3 lucruri: `fn` (funcția), `inputs`, `outputs`.

In [2]:
def saluta(nume):
    return "Salut, " + nume

gr.Interface(fn=saluta, inputs="text", outputs="text").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Atât. Caseta, butonul *Submit*, totul l-a făcut Gradio. Pe asta se construiește
toată aplicația.

## 2. Mai multe input-uri = o listă

Tab-urile noastre au mai multe câmpuri. Dacă funcția are mai multe argumente,
dai o **listă** la `inputs` (ordinea = ordinea argumentelor).

In [3]:
def combina(text, optiune, numar):
    return f"[{optiune} @ {numar}] {text}"

gr.Interface(
    fn=combina,
    inputs=[gr.Textbox(label="Text"),
            gr.Dropdown(["a", "b"], value="a", label="Opțiune"),
            gr.Slider(0, 1, value=0.3, step=0.1, label="Număr")],
    outputs=gr.Textbox(label="Rezultat"),
).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Reține tiparul `[text, dropdown, slider] -> funcție -> text`. **Asta e un tab.**

## 3. Backend fals (ca să rulăm fără chei API)

În `app.py` real, sus, sunt 2 importuri din `core/` (construite în C6–C7):

```python
from core.agent import generate_agent_response   # un agent RAG
from core.graph import run_thread                 # dezbatere multi-agent
```

Aici le înlocuim cu funcții-jucărie. Restul codului rămâne identic ca structură.

In [5]:
def fake_llm(prompt):
    return "(răspuns simulat) despre: " + prompt[:70]

def fake_agent(slug, stimulus):
    voci = {"anti_sistem": "Instituțiile par din nou rupte de oameni.",
            "pro_european": "Să discutăm pe baza procedurilor."}
    return voci.get(slug, f"[{slug}] {stimulus[:50]}")

AGENTS = [("Anti-sistem", "anti_sistem"), ("Pro-european", "pro_european")]
print("backend fals pregătit")

backend fals pregătit


## 4. Primul tab real: Chat

În `app.py`, tab-ul Chat e funcția `chat()` + un `gr.Interface`. O reproducem
cu `fake_llm`.

In [6]:
def chat(prompt):
    return fake_llm(prompt) if prompt.strip() else "Scrie un prompt."

gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label="Întrebare / prompt", lines=4),
    outputs=gr.Textbox(label="Răspuns", lines=10),
    title="Chat",
).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Tab-ul Chat complet, fără să scriem vreun buton. Gradio l-a făcut.

## 5. Starea partajată: `CFG` și `ART`

Tab-ul **Setări** alege modelul și (opțional) încarcă o știre. Celelalte tab-uri
trebuie să **vadă** acea știre. Soluția minimă: două dicționare la nivel de modul.

- `CFG` = provider / model / temperatură
- `ART` = textul + titlul știrii încărcate

Setări **scrie** în ele, restul tab-urilor **citesc**. (Alternativa „canonică"
ar fi `gr.State`; varianta minimă alege simplitatea.)

Truc din `app.py`: provider + model sunt **un singur dropdown**
(`"provider|model"`) — imposibil să fie nepotrivite.

In [7]:
CFG = {"provider": "gemini", "model": "gemini-2.5-flash-lite", "temp": 0.3}
ART = {"text": "", "title": ""}

MODEL_CHOICES = [("gemini · gemini-2.5-flash-lite", "gemini|gemini-2.5-flash-lite"),
                 ("deepseek · deepseek-chat",       "deepseek|deepseek-chat")]

def setup(model_choice, temperature, fake_url):
    provider, model = model_choice.split("|", 1)     # despărțim "provider|model"
    CFG.update(provider=provider, model=model, temp=temperature)
    if fake_url.strip():
        ART.update(text=f"Text fals al știrii de la {fake_url}", title=fake_url)
        return f"Setări salvate. Știre ACTIVĂ: {fake_url}"
    ART.update(text="", title="")
    return f"Setări salvate ({provider} · {model}). Fără știre."

gr.Interface(
    fn=setup,
    inputs=[gr.Dropdown(MODEL_CHOICES, value=MODEL_CHOICES[0][1],
                        label="Provider · Model"),
            gr.Slider(0, 1, value=0.3, step=0.1, label="Temperatură"),
            gr.Textbox(label="URL știre (gol = fără știre)")],
    outputs=gr.Textbox(label="Stare"),
    title="Setări",
).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## 6. Regula cheie: subiectul intră *peste* știre

`_subject()` decide ce primește un agent:

- **știre + subiect** -> vorbim despre subiect, dar **în contextul știrii**
- **doar știre** -> vorbim despre știre
- **doar subiect** -> vorbim doar despre subiect

In [8]:
def _subject(typed):
    typed = (typed or "").strip()
    news = ART["text"].strip()
    if news and typed:
        return f"{typed}\n\n[În contextul acestei știri:]\n{news[:600]}"
    if news:
        return news[:700]
    return typed

ART.update(text="Știre despre UE și energie.")
print(_subject("Bolojan"))     # subiect peste știre
ART.update(text="")
print(_subject("Bolojan"))     # doar subiect

Bolojan

[În contextul acestei știri:]
Știre despre UE și energie.
Bolojan


## 7. Tab-ul Agent

Tab-ul Agent = `_subject()` + chemarea backend-ului. În `app.py` real,
`fake_agent` e `generate_agent_response` din `core.agent` (C6).

In [9]:
def agent(text, slug):
    s = _subject(text)
    if not s.strip():
        return "Încarcă o știre sau scrie un subiect."
    return fake_agent(slug, s)               # în app: generate_agent_response(...)

gr.Interface(
    fn=agent,
    inputs=[gr.Textbox(label="Subiect (intră peste știre, dacă e încărcată)",
                       lines=3),
            gr.Dropdown(AGENTS, value="anti_sistem", label="Agent")],
    outputs=gr.Textbox(label="Comentariu", lines=10),
    title="Agent",
).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Tab-urile **Rezumat**, **Toți agenții** și **Dezbatere** au exact același tipar:
o funcție + un `gr.Interface`. Doar funcția diferă (rezumat / loop pe roluri /
`core.graph.run_thread`).

## 8. Punem tab-urile împreună

`app.py` are 6 tab-uri cu o **temă comună**. `gr.TabbedInterface` nu acceptă
`theme=` pe toate versiunile, așa că facem ce face el intern: un `gr.Blocks`
cu temă, `gr.Tabs`, și randăm fiecare `Interface` cu `.render()`.

In [10]:
tab_setup = gr.Interface(setup,
    [gr.Dropdown(MODEL_CHOICES, value=MODEL_CHOICES[0][1], label="Provider · Model"),
     gr.Slider(0, 1, value=0.3, step=0.1, label="Temperatură"),
     gr.Textbox(label="URL știre")],
    gr.Textbox(label="Stare"), title="Setări")

tab_chat = gr.Interface(chat, gr.Textbox(label="Prompt", lines=3),
    gr.Textbox(label="Răspuns", lines=8), title="Chat")

tab_agent = gr.Interface(agent,
    [gr.Textbox(label="Subiect", lines=3),
     gr.Dropdown(AGENTS, value="anti_sistem", label="Agent")],
    gr.Textbox(label="Comentariu", lines=8), title="Agent")

TABS = [("Setări", tab_setup), ("Chat", tab_chat), ("Agent", tab_agent)]

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown("# EchoChamber Studio")
    with gr.Tabs():
        for nume, iface in TABS:
            with gr.Tab(nume):
                iface.render()

demo.launch()

/var/folders/fr/39tnf43d29z8nlk6875jbzq80000gn/T/ipykernel_30994/3885253093.py:17: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Acesta e scheletul exact din `app.py`. În aplicația reală sunt 6 tab-uri în
loc de 3, iar funcțiile cheamă `core/` în loc de `fake_*`.

## 9. Recapitulare

**Ce face Gradio (tot tutorialul, pe scurt):**

1. `gr.Interface(fn, inputs, outputs)` — o funcție devine interfață.
2. Mai multe input-uri = o listă. Un astfel de bloc = un tab.
3. `CFG` / `ART` (dict-uri de modul) = starea partajată: Setări scrie, restul citesc.
4. `_subject()` = regula subiect-peste-știre.
5. `gr.Blocks` + `gr.Tabs` + `.render()` = cele 6 tab-uri cu temă comună.

**Dependențe (din structura repo):** `app/app.py` cheamă doar `core/` —
nu rescrie nimic.

| Tab(uri) | Funcție în app.py | Backend | Curs |
|---|---|---|---|
| Setări · Chat · Rezumat | `setup` · `chat` · `summary` | apel LLM direct | C2 |
| Agent | `agent` -> `_agent` | `core.agent` (FAISS + rol) | C5 + C6 |
| Toți agenții | `all_agents` | loop pe `roles.yaml` -> `core.agent` | C6 |
| Dezbatere | `debate` | `core.graph.run_thread` (LangGraph) | C7 |

`core.agent` -> `core.retriever` (FAISS) + `roles.yaml` + LLM.
`core.graph` orchestrează `core.agent` (round-robin). Singura punte offline->runtime:
vectorstore-urile construite offline, citite de retriever la fiecare cerere.

**Mesajul cheie:** aplicația nu e un proiect nou. E un strat subțire Gradio
peste funcțiile din C2–C7. Fiecare tab = un buton peste o funcție de curs.

## Tema 3 - Extensie individuală student_03


In [ ]:
import gradio as gr
import re

# -----------------------------
# Funcții noi pentru Tema 3
# -----------------------------

def clean_text(text):
    """
    Curăță textul introdus de utilizator:
    - elimină spațiile multiple
    - elimină spațiile de la început și final
    """
    if not text:
        return ""
    
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text


def text_statistics(text):
    """
    Calculează statistici simple pentru textul introdus.
    """
    cleaned = clean_text(text)
    
    if not cleaned:
        return "Nu ai introdus niciun text."
    
    words = cleaned.split()
    characters = len(cleaned)
    
    return f"""
### Statistici text

- Număr de cuvinte: **{len(words)}**
- Număr de caractere: **{characters}**
- Text curățat:  
{cleaned}
"""


def demo_agent_response(text, tone, response_length):
    """
    Funcție simplă pentru testarea interfeței.
    În notebook-ul real, aici poate fi conectată funcția de generare a răspunsului agentului.
    """
    cleaned = clean_text(text)
    
    if not cleaned:
        return "Te rog introdu un text sau o știre pentru analiză."
    
    return f"""
### Răspuns agent - simulare interfață

Agentul a primit următorul input:

> {cleaned}

**Ton selectat:** {tone}  
**Lungime răspuns:** {response_length}

Acesta este un exemplu de răspuns formatat. În aplicația completă, aici ar fi afișat răspunsul generat de agentul EchoChamber.
"""


# -----------------------------
# Interfața Gradio extinsă
# -----------------------------

with gr.Blocks(
    title="EchoChamber - Tema 3 student_03",
    theme=gr.themes.Soft()
) as demo:
    
    gr.Markdown(
        """
        # 🗣️ EchoChamber - Gradio Mini App
        
        Această interfață arată cum se leagă un input text, o funcție Python și un output vizibil în Gradio.
        
        **Extensie individuală student_03:** am adăugat o funcție nouă, o opțiune pentru utilizator, un tab nou și mici modificări de design.
        """
    )
    
    with gr.Tab("🤖 Agent response"):
        gr.Markdown("### Testare răspuns agent")
        
        with gr.Row():
            with gr.Column():
                user_input = gr.Textbox(
                    label="Introdu o știre sau o afirmație politică",
                    placeholder="Exemplu: CCR a decis anularea alegerilor după suspiciuni privind influențe externe.",
                    lines=5
                )
                
                tone_dropdown = gr.Dropdown(
                    choices=["Neutru", "Critic", "Explicativ"],
                    value="Explicativ",
                    label="Alege tonul răspunsului"
                )
                
                length_radio = gr.Radio(
                    choices=["Scurt", "Mediu", "Lung"],
                    value="Mediu",
                    label="Alege lungimea răspunsului"
                )
                
                generate_btn = gr.Button("Generează răspuns")
            
            with gr.Column():
                agent_output = gr.Markdown(label="Răspuns generat")
        
        generate_btn.click(
            fn=demo_agent_response,
            inputs=[user_input, tone_dropdown, length_radio],
            outputs=agent_output
        )
    
    
    with gr.Tab("🧹 Curățare text"):
        gr.Markdown("### Funcție nouă: curățare și statistici text")
        
        stats_input = gr.Textbox(
            label="Text pentru analiză",
            placeholder="Lipește aici un comentariu sau o știre...",
            lines=5
        )
        
        stats_btn = gr.Button("Curăță textul și calculează statistici")
        stats_output = gr.Markdown()
        
        stats_btn.click(
            fn=text_statistics,
            inputs=stats_input,
            outputs=stats_output
        )
    
    
    with gr.Tab("ℹ️ Ajutor & Etică"):
        gr.Markdown(
            """
            ## Despre interfață
            
            Această aplicație este o versiune de test pentru proiectul EchoChamber.
            
            Scopul ei este să arate cum poate fi conectat un text introdus de utilizator cu o funcție Python și cu un răspuns afișat în interfață.
            
            ## Observație etică
            
            Răspunsurile generate de agenți nu trebuie considerate adevăr absolut.  
            Înainte ca un astfel de sistem să fie folosit public, rezultatele trebuie verificate de un om, mai ales în contexte politice sau sensibile.
            """
        )

demo.launch()

/var/folders/fr/39tnf43d29z8nlk6875jbzq80000gn/T/ipykernel_30994/302219967.py:72: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


### Explicație Tema 3

What I added: Am adăugat o extensie individuală pentru interfața Gradio, cu taburi noi și opțiuni vizibile pentru utilizator.

New function created: Am creat funcțiile `clean_text()` și `text_statistics()`, care curăță textul introdus și calculează numărul de cuvinte și caractere.

Design/interface change: Am modificat interfața printr-un titlu mai clar, emoji-uri, taburi separate, dropdown pentru ton și radio button pentru lungimea răspunsului.

What I would improve next: Aș conecta direct funcția de generare a răspunsului agentului real din `core/agent.py`, astfel încât aplicația să nu afișeze doar un exemplu, ci răspunsul complet al agentului EchoChamber.